In [ ]:
# Full ML Cycle Habit Tracker (run this at the start of each project)

project_name = "SET_THIS_PROJECT_NAME"

cycle_steps = [
    "1_problem_definition",
    "2_data_collection_and_scope",
    "3_data_transformation_and_processing",
    "4_baseline_model_or_system",
    "5_evaluation_metrics_and_results",
    "6_error_analysis",
    "7_improvement_iteration",
    "8_deployment_or_handoff_notes",
]

status = {step: "TODO" for step in cycle_steps}
evidence = {step: "" for step in cycle_steps}

def mark_done(step, note=""):
    if step not in status:
        raise ValueError(f"Unknown step: {step}")
    status[step] = "DONE"
    evidence[step] = note

def show_progress():
    done_count = sum(1 for v in status.values() if v == "DONE")
    total = len(status)
    print(f"\nProject: {project_name}")
    print(f"Progress: {done_count}/{total} steps complete")
    for step in cycle_steps:
        note = f" | note: {evidence[step]}" if evidence[step] else ""
        print(f"- {step}: {status[step]}{note}")

# Example usage:
# mark_done("3_data_transformation_and_processing", "Handled missing values and encoded categories")
# mark_done("5_evaluation_metrics_and_results", "F1=0.81, recall=0.86")
show_progress()

# Full Process Lens For This Project

Use this section every time you start a project so you practice the same end-to-end workflow, not just isolated coding tasks.

## 1) Data Transformation and Processing (What and Why)

Raw data is rarely model-ready. Your first job is to transform data into a reliable, learnable format.

What to do in every project:
- Identify data types and expected schema.
- Handle missing values, duplicates, and inconsistent formats.
- Convert features into model-usable representations (encoding, scaling, tokenization, chunking, etc.).
- Keep transformations reproducible so train and inference use the same logic.
- Document assumptions and risks introduced by preprocessing choices.

Why this matters:
- Better preprocessing usually improves results more than switching algorithms.
- Poor preprocessing creates hidden errors that look like model failure.

## 2) Evaluating and Improving Models (What and Why)

Evaluation is not the last step. It is the loop that drives improvement.

What to do in every project:
- Start with a baseline and compare against it.
- Choose metrics tied to the real goal (not just convenience metrics).
- Inspect errors by slice (segments, classes, edge cases).
- Tune thresholds, features, prompts, retrieval settings, or hyperparameters based on evidence.
- Re-evaluate after each change and keep track of what improved and what regressed.

Why this matters:
- A model can appear good overall but fail on important cases.
- Iterative evaluation is how projects become production-ready, not just demo-ready.

## 3) Project Reflection Checklist

Before marking this project complete, confirm:
- I can explain how data was transformed and why.
- I can explain which metrics I chose and why.
- I can show at least one improvement cycle from evaluation findings.
- I can describe current limitations and next improvements.

# Week 2 — pandas Deep Dive
> **Phase 0 | Foundation** — The most important library for data science.

---

## Beginner Start Here

**pandas** is the #1 tool for working with tabular data (tables, spreadsheets, CSVs). You will use it in 100% of ML projects. This week is deep and practical — by the end you'll be able to load, explore, clean, transform, and summarize any dataset.

### What You Will Learn
- What a Series and DataFrame are
- Loading CSV data and inspecting it
- Selecting rows and columns
- Filtering data with conditions
- Handling missing values
- Grouping and aggregating data
- Sorting, renaming, and adding columns
- Saving cleaned data back to CSV

### Key Terms
| Term | Plain English |
|------|---------------|
| **DataFrame** | A 2D table with labeled rows and columns — like a spreadsheet in Python |
| **Series** | A single column — a 1D array with an index |
| **Index** | The row labels (usually 0, 1, 2... but can be anything) |
| **Column** | A named variable in the dataset — each column is a Series |
| **NaN** | Not a Number — pandas' way of representing a missing value |
| **dtype** | The data type of a column (int64, float64, object=string, bool) |
| **Aggregation** | Summarizing many rows to one value (sum, mean, count, max) |
| **GroupBy** | Split the data into groups, then apply a function to each group |
| **Mask / Filter** | A boolean condition used to select specific rows |

In [ ]:
# ── First run check + imports ─────────────────────────────────────────────────
import pandas as pd
import numpy as np

print(f"✅ pandas {pd.__version__} loaded. Week 2 notebook ready.")
pd.set_option('display.max_columns', 20)    # show up to 20 columns
pd.set_option('display.float_format', '{:.2f}'.format)  # 2 decimal places

---

## Section 1: Series — The Building Block

A **Series** is a one-dimensional labeled array — think of it as a single column from a spreadsheet.

Every DataFrame column is a Series. When you select one column from a DataFrame you get a Series back.

In [ ]:
# ── Series ────────────────────────────────────────────────────────────────────

# Create a Series from a list
charges = pd.Series([65.0, 79.99, 45.0, 89.99, 55.0, 79.99, 99.0])
print("Basic Series:")
print(charges)
print(f"\nType: {type(charges)}")
print(f"dtype: {charges.dtype}  ← numeric (float64)")

# Series with custom index
customer_ids = ["C001", "C002", "C003", "C004", "C005", "C006", "C007"]
charges_indexed = pd.Series([65.0, 79.99, 45.0, 89.99, 55.0, 79.99, 99.0], index=customer_ids)
print("\nSeries with custom index:")
print(charges_indexed)

# Accessing values
print(f"\nCustomer C003's charges: ${charges_indexed['C003']}")
print(f"First element (position 0): ${charges_indexed.iloc[0]}")

# Statistics on a Series
print(f"\nMean:   ${charges.mean():.2f}")
print(f"Median: ${charges.median():.2f}")
print(f"Std:    ${charges.std():.2f}")
print(f"\nFull describe():")
print(charges.describe())

---

## Section 2: DataFrame — The Core Tool

A **DataFrame** is a 2D table — rows and columns. It's the pandas equivalent of a spreadsheet or SQL table.

You'll usually create a DataFrame by loading a CSV file: `pd.read_csv('file.csv')`

In [ ]:
# ── Create a DataFrame ────────────────────────────────────────────────────────
# In real projects you'd load a CSV. Here we build one from scratch for learning.

data = {
    "customer_id":      ["C001", "C002", "C003", "C004", "C005", "C006", "C007", "C008"],
    "tenure_months":    [12, 3, 36, 6, 24, 2, 48, 1],
    "monthly_charges":  [65.0, 110.0, 45.0, 89.99, 55.0, 79.99, 35.0, 99.0],
    "contract":         ["Month-to-month", "Month-to-month", "Two year", "One year",
                         "Two year", "Month-to-month", "Two year", "Month-to-month"],
    "has_internet":     [True, True, False, True, True, True, False, True],
    "support_tickets":  [1, 5, 0, 3, 0, np.nan, 1, 8],  # np.nan = missing!
    "churned":          [0, 1, 0, 1, 0, 1, 0, 1],
}

df = pd.DataFrame(data)
print(df)
print(f"\nShape: {df.shape}  ← ({df.shape[0]} rows, {df.shape[1]} columns)")

In [ ]:
# ── Inspecting a DataFrame ────────────────────────────────────────────────────
# These are the FIRST things you run whenever you open a new dataset.

print("=" * 50)
print("df.head(3) — first 3 rows:")
print("=" * 50)
print(df.head(3))

print("\n" + "=" * 50)
print("df.info() — column types and missing values:")
print("=" * 50)
df.info()

print("\n" + "=" * 50)
print("df.describe() — numeric stats summary:")
print("=" * 50)
print(df.describe())

print("\n" + "=" * 50)
print("df.dtypes — column data types:")
print("=" * 50)
print(df.dtypes)

---

## Section 3: Selecting Columns and Rows

Three ways to select data:
- `df['column']` — select one column (returns a Series)
- `df[['col1', 'col2']]` — select multiple columns (returns a DataFrame)
- `df.loc[row_label, col_label]` — select by label
- `df.iloc[row_number, col_number]` — select by position (index number)

In [ ]:
# ── Column Selection ──────────────────────────────────────────────────────────

# Single column → Series
charges_col = df['monthly_charges']
print(f"Single column (returns Series):")
print(charges_col.values)
print(f"type: {type(charges_col).__name__}")

# Multiple columns → DataFrame
features = df[['tenure_months', 'monthly_charges', 'churned']]
print(f"\nMultiple columns (returns DataFrame):")
print(features)

# Access all column names
print(f"\nColumn names: {list(df.columns)}")

In [ ]:
# ── Row Selection: loc vs iloc ────────────────────────────────────────────────

# .loc — select by index LABEL
print(".loc[2] — row at label/index 2:")
print(df.loc[2])

# .iloc — select by position NUMBER
print("\n.iloc[0] — first row (position 0):")
print(df.iloc[0])

# Select a range of rows
print("\n.iloc[0:3] — first 3 rows:")
print(df.iloc[0:3])

# Select a specific cell: row label 1, column 'monthly_charges'
print(f"\nCell at row 1, 'monthly_charges': ${df.loc[1, 'monthly_charges']}")

# Select row 2, columns 0 and 1 (by position)
print(f"Cell at row 2, col position 2: {df.iloc[2, 2]}")

---

## Section 4: Filtering Rows (Boolean Masking)

The most powerful way to select rows. Create a condition, get a column of True/False, then use it to filter.

```python
mask = df['monthly_charges'] > 75    # Series of True/False
df[mask]                              # only rows where mask is True
# Same thing in one line:
df[df['monthly_charges'] > 75]
```

In [ ]:
# ── Boolean Filtering ─────────────────────────────────────────────────────────

# Single condition
high_value = df[df['monthly_charges'] > 75]
print(f"Customers with charges > $75: {len(high_value)} rows")
print(high_value[['customer_id', 'monthly_charges', 'churned']])

# Multiple conditions: use & (and) and | (or) — NOT Python's 'and'/'or'
at_risk = df[(df['monthly_charges'] > 75) & (df['churned'] == 1)]
print(f"\nHigh charges AND churned: {len(at_risk)} customers")
print(at_risk[['customer_id', 'monthly_charges', 'tenure_months', 'churned']])

# Filter with string match
month_to_month = df[df['contract'] == 'Month-to-month']
print(f"\nMonth-to-month contracts: {len(month_to_month)} customers")

# .isin() — match against a list of values
short_contracts = df[df['contract'].isin(['Month-to-month', 'One year'])]
print(f"Short contracts (M-to-M or 1yr): {len(short_contracts)} customers")

# Negation with ~
long_tenure = df[~(df['tenure_months'] < 12)]  # NOT short tenure
print(f"Tenure >= 12 months: {len(long_tenure)} customers")

In [ ]:
# ── YOUR TURN: Filtering ──────────────────────────────────────────────────────
# 1. Select all churned customers (churned == 1)
# 2. Select all customers with tenure < 6 months AND monthly charges > $80
# 3. Print the average monthly_charges for churned vs non-churned customers

# YOUR CODE HERE

---

## Section 5: Handling Missing Values

Real data is messy. Missing values are represented as `NaN` (Not a Number) in pandas. Before training any ML model, you **must** handle missing values — most algorithms can't work with them.

**Three strategies:**
1. **Drop** rows or columns with missing values (quick but loses data)
2. **Fill** with a statistic: mean, median, or mode
3. **Fill** with a special value (e.g., 0, -1, or 'unknown')

In [ ]:
# ── Missing Values ────────────────────────────────────────────────────────────

# Step 1: Find missing values
print("Missing values per column:")
print(df.isnull().sum())

print(f"\nTotal missing: {df.isnull().sum().sum()}")
print(f"Missing as percentage:")
print((df.isnull().sum() / len(df) * 100).round(1))

# Step 2: Inspect the rows with missing values
rows_with_missing = df[df['support_tickets'].isnull()]
print(f"\nRows with missing support_tickets:")
print(rows_with_missing)

# Step 3: Fill missing values
df_cleaned = df.copy()  # always work on a copy!

# Strategy A: fill with median (robust to outliers — better than mean)
median_tickets = df['support_tickets'].median()
df_cleaned['support_tickets'] = df_cleaned['support_tickets'].fillna(median_tickets)
print(f"\nFilled NaN with median ({median_tickets})")
print(f"Missing after fill: {df_cleaned['support_tickets'].isnull().sum()}")

# Strategy B: drop rows with missing values
df_dropped = df.dropna()
print(f"\nAfter dropna(): {len(df_dropped)} rows (was {len(df)})")

---

## Section 6: Adding and Transforming Columns

In [ ]:
# ── Adding and Transforming Columns ──────────────────────────────────────────

df2 = df_cleaned.copy()

# Calculate total charges
df2['total_charges'] = df2['monthly_charges'] * df2['tenure_months']

# Bin tenure into categories
df2['tenure_band'] = pd.cut(
    df2['tenure_months'],
    bins=[0, 6, 12, 24, 100],
    labels=['0-6m', '7-12m', '13-24m', '24m+']
)

# Encode contract as number (for ML)
contract_map = {"Month-to-month": 0, "One year": 1, "Two year": 2}
df2['contract_encoded'] = df2['contract'].map(contract_map)

# Rename a column
df2 = df2.rename(columns={'support_tickets': 'tickets'})

print("Updated DataFrame:")
print(df2[['customer_id', 'tenure_months', 'tenure_band', 'total_charges', 'contract_encoded']].to_string())
print(f"\nNew columns: {list(df2.columns)}")

---

## Section 7: GroupBy — Split, Apply, Combine

**GroupBy** is one of the most powerful tools in pandas. It lets you:
1. **Split** the data into groups (e.g., by `contract` type)
2. **Apply** a function to each group (e.g., mean of `monthly_charges`)
3. **Combine** the results back into a table

This is how you answer questions like: "Do churned customers pay more on average?"

In [ ]:
# ── GroupBy ───────────────────────────────────────────────────────────────────

# Average monthly charges by churn status
print("Average charges by churn status:")
print(df2.groupby('churned')['monthly_charges'].mean())
print("(0 = not churned, 1 = churned)")

# Multiple stats at once with .agg()
print("\nDetailed stats by churn status:")
summary = df2.groupby('churned').agg(
    count=('customer_id', 'count'),
    avg_charges=('monthly_charges', 'mean'),
    avg_tenure=('tenure_months', 'mean'),
    avg_tickets=('tickets', 'mean')
).round(2)
print(summary)

# GroupBy on a string column
print("\nAverage charges and churn rate by contract type:")
contract_summary = df2.groupby('contract').agg(
    n_customers=('customer_id', 'count'),
    avg_monthly_charges=('monthly_charges', 'mean'),
    churn_rate=('churned', 'mean')  # mean of 0/1 = proportion that churned
).round(3)
print(contract_summary)

In [ ]:
# ── YOUR TURN: GroupBy ────────────────────────────────────────────────────────
# 1. Using df2, group by 'tenure_band' and calculate:
#    - number of customers in each band
#    - average monthly_charges
#    - churn rate (mean of churned column)
# 2. Print the results
# 3. Which tenure band has the highest churn rate?

# YOUR CODE HERE

---

## Section 8: Sorting and Value Counts

In [ ]:
# ── Sorting ───────────────────────────────────────────────────────────────────

# Sort by one column (descending)
print("Top 3 highest-paying customers:")
print(df2.sort_values('monthly_charges', ascending=False).head(3)[['customer_id', 'monthly_charges', 'churned']])

# Sort by multiple columns
print("\nSorted by churned (desc) then monthly_charges (desc):")
print(df2.sort_values(['churned', 'monthly_charges'], ascending=[False, False])[['customer_id', 'churned', 'monthly_charges']].to_string())

# Value counts — distribution of categorical column
print("\nContract type distribution:")
print(df2['contract'].value_counts())

print("\nChurn distribution (with percentages):")
print(df2['churned'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

---

## Section 9: Full EDA Workflow on a Larger Dataset

In [ ]:
# ── Generate a realistic fake dataset ────────────────────────────────────────
# We'll create 200 rows with realistic churn patterns to practice on.

np.random.seed(42)
n = 200

tenure = np.random.randint(1, 60, n)
charges = np.random.uniform(30, 120, n).round(2)
contract_types = np.random.choice(["Month-to-month", "One year", "Two year"],
                                   n, p=[0.55, 0.25, 0.20])

# Churn probability: high charges + short tenure + M-to-M → high churn
churn_prob = 0.3 + 0.3*(charges > 85) + 0.3*(tenure < 12) - 0.2*(contract_types == 'Two year')
churn_prob = np.clip(churn_prob, 0, 1)
churned = (np.random.rand(n) < churn_prob).astype(int)

big_df = pd.DataFrame({
    'tenure_months': tenure,
    'monthly_charges': charges,
    'contract': contract_types,
    'churned': churned
})
# Introduce some missing values
missing_idx = np.random.choice(big_df.index, size=15, replace=False)
big_df.loc[missing_idx, 'monthly_charges'] = np.nan

print(f"Dataset shape: {big_df.shape}")
print(big_df.head())

In [ ]:
# ── Full EDA in one cell ──────────────────────────────────────────────────────

print("=" * 60)
print("1. SHAPE & DTYPES")
print("=" * 60)
print(big_df.dtypes)

print("\n" + "=" * 60)
print("2. MISSING VALUES")
print("=" * 60)
print(big_df.isnull().sum())

print("\n" + "=" * 60)
print("3. SUMMARY STATS")
print("=" * 60)
print(big_df.describe())

print("\n" + "=" * 60)
print("4. TARGET DISTRIBUTION")
print("=" * 60)
churn_counts = big_df['churned'].value_counts()
for val, cnt in churn_counts.items():
    label = "Churned" if val == 1 else "Active"
    pct = cnt / len(big_df) * 100
    print(f"  {label}: {cnt} ({pct:.1f}%)")

print("\n" + "=" * 60)
print("5. CHURN RATE BY CONTRACT TYPE")
print("=" * 60)
print(big_df.groupby('contract')['churned'].agg(['mean', 'count']).round(3))

In [ ]:
# ── Data Cleaning Pipeline ────────────────────────────────────────────────────

def clean_data(df):
    """Standard cleaning pipeline for the churn dataset."""
    df = df.copy()
    
    # 1. Fill missing monthly_charges with median
    df['monthly_charges'] = df['monthly_charges'].fillna(df['monthly_charges'].median())
    
    # 2. Calculate total_charges
    df['total_charges'] = df['monthly_charges'] * df['tenure_months']
    
    # 3. Encode contract
    contract_map = {"Month-to-month": 0, "One year": 1, "Two year": 2}
    df['contract_enc'] = df['contract'].map(contract_map)
    
    # 4. Flag high-risk customers
    df['high_risk'] = ((df['monthly_charges'] > 85) & (df['tenure_months'] < 12)).astype(int)
    
    return df

clean = clean_data(big_df)

print(f"After cleaning:")
print(f"  Missing values: {clean.isnull().sum().sum()}")
print(f"  High-risk customers: {clean['high_risk'].sum()}")
print(clean.head())

In [ ]:
# ── Saving to CSV ─────────────────────────────────────────────────────────────

# Save cleaned data
import os
os.makedirs('../data', exist_ok=True)   # create data/ folder if it doesn't exist

clean.to_csv('../data/churn_cleaned.csv', index=False)
print("✅ Saved to data/churn_cleaned.csv")

# Reload and verify
reloaded = pd.read_csv('../data/churn_cleaned.csv')
print(f"Reloaded: {reloaded.shape}")
print(reloaded.head(2))

---

## Week 2 Reflection Questions

**Answer these (double-click to edit):**

1. What is the difference between a Series and a DataFrame?

2. What does `df.isnull().sum()` tell you?

3. Why should you fill missing values with the median rather than the mean?

4. What does `df.groupby('churn')['charges'].mean()` do in plain English?

5. What's the difference between `.loc[]` and `.iloc[]`?

---

## Week 2 Checklist — Before You Move to Week 3

- [ ] I can create a DataFrame from a dictionary
- [ ] I can run `head()`, `info()`, `describe()` and understand the output
- [ ] I can select single and multiple columns
- [ ] I can filter rows with boolean conditions using `&` and `|`
- [ ] I can find and fill missing values
- [ ] I can add new columns using math on existing columns
- [ ] I can group data with `groupby()` and apply `mean()` or `agg()`
- [ ] I ran every code cell and completed all YOUR TURN exercises
- [ ] I saved a cleaned DataFrame to CSV and reloaded it

**If you checked all boxes → go to STARTER_Week3_NumPyDeep.ipynb**